# Dense+BM25+RRF와 Qwen 평가

하이브리드 검색과 재순위화, Qwen2.5-1.5B 채점 실험. 평가 방식의 한계를 함께 기록합니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


## 준비. 패키지 설치와 드라이브 연결


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" "autoawq==0.2.9" accelerate rank_bm25


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()


df = pd.read_csv(metadata_csv, encoding="utf-8")
loaded_files = []

for name in os.listdir(folder):
    full_path = os.path.join(folder, name)

    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = loader.load()[0].page_content

    loaded_files.append((name, text))

print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]

documents = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        continue

    row = result_df.iloc[0]
    metadata = {column: row[column] for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))

print("메타데이터와 연결한 문서:", len(documents))


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 후 FAISS DB에 저장

`embedding_model_name`을 바꾸면 임베딩 모델을 조정할 수 있습니다.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model_name = "Alibaba-NLP/gte-multilingual-base"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embeddings)
print("저장한 청크:", vector_store.index.ntotal)


## 6. 질문 받기


In [ ]:
evaluation_data = [
    {
        "query": "벤처확인종합관리시스템 기능 고도화 용역의 사업기간과 예산은 얼마인가요?",
        "checks": ["사업기간", "예산"],
    },
    {
        "query": "벤처확인종합관리시스템 기능 고도화 용역의 기술평가와 가격평가 비율은 각각 얼마인가요?",
        "checks": ["기술평가 비율", "가격평가 비율"],
    },
    {
        "query": "BIFF & ACFM 온라인서비스 재개발 사업의 기간과 예산은 얼마인가요?",
        "checks": ["사업기간", "예산"],
    },
    {
        "query": "BIFF & ACFM 온라인서비스 사업에서 BIFF와 ACFM에 각각 어떤 인력을 상주 파견하나요?",
        "checks": ["BIFF 상주 파견 인력", "ACFM 상주 파견 인력"],
    },
    {
        "query": "KUSF 체육특기자 경기기록 관리시스템은 1차 연도에 어느 종목을 우선하고, 이후 어떤 종목으로 확대하나요?",
        "checks": ["1차 연도 우선 종목", "이후 확대 종목"],
    },
    {
        "query": "예술경영지원센터 통합 정보시스템 구축 사전 컨설팅의 제안서는 어떻게 제출하며, 기한이 지난 서류는 인정되나요?",
        "checks": ["제안서 제출 방법", "기한이 지난 서류의 인정 여부"],
    },
]

# checks는 확인할 항목 이름이며, 실제 정답을 뜻하지 않습니다.
question = evaluation_data  # 기존 RAG 반복문 유지


## 7. 하이브리드 검색 (Dense + BM25, RRF)


In [ ]:
from langchain_community.retrievers import BM25Retriever

dense_retriever = vector_store.as_retriever(search_kwargs={"k": 8})
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 8


def rrf_search(query, k=60):
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 8. HyDE로 검색 문장 만들기


In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


language_model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"
model_revision = "f2699f35d5b4ba511ab93826c22bd07370296b5c"


model = AutoModelForCausalLM.from_pretrained(
    language_model_name, torch_dtype=torch.float16,
    trust_remote_code=True, device_map="auto", revision=model_revision
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name, revision=model_revision)


llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=220,
)


llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)

method = "hyde"  # "basic" 또는 "hyde"


def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")


## 9. 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device="cpu"
)

# 확인 셀과 답변 생성 셀이 같은 검색·재순위화 결과를 공유합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method):
    cache_key = (method, query)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text)
        ranking = reranker.rank(
            query,  # 재순위화에는 원래 질문 사용
            [doc.page_content for doc in found],
            top_k=4,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (search_text, found, reranked_docs)
    return retrieval_cache[cache_key]


for i, q in enumerate(question, start=1):
    query = q["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    print(f"\n{i}번째 질문: {query}")
    print("검색 문장:", search_text)
    print("리랭크 전:", [doc.metadata["chunk_id"] for doc in found])
    print("리랭크 후:", [doc.metadata["chunk_id"] for doc in reranked_docs])


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

## 10. 답변 생성


In [ ]:
# 위 셀의 검색·재순위화 결과로 최종 답변을 한 번만 생성합니다.
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''

answers = []
for i, item in enumerate(evaluation_data, start=1):
    query = item["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    answer = chat_model.invoke(
        answer_rules.format(context=context, query=query)
    ).content.strip()

    answers.append({
        "query": query,
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
    })
    print(f"{i}번째 질문: {query}")
    print("사용 청크:", answers[-1]["chunk_ids"])
    print("답변:", answer, "\n")


## 11. 평가

위에서 만든 `answers`의 청크와 답변을 채점합니다.


In [ ]:
# 위에서 생성해 저장한 answers만 평가합니다. 여기서는 답변을 다시 생성하지 않습니다.
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if "answers" not in globals() or len(answers) != len(evaluation_data):
    raise ValueError("먼저 바로 위의 답변 생성 셀을 실행해 주세요.")

judge_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
if "judge_model" not in globals() or "judge_tokenizer" not in globals():
    judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_id)
    judge_model = AutoModelForCausalLM.from_pretrained(
        judge_model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    ).eval()

# 이 모델의 기본 sampling 설정은 do_sample=False에서 쓰이지 않습니다.
judge_model.generation_config.temperature = None
judge_model.generation_config.top_p = None
judge_model.generation_config.top_k = None

score_tokens = {
    score: judge_tokenizer.encode(score, add_special_tokens=False)
    for score in ("0", "0.5", "1")
}


def judge_score(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = judge_tokenizer(
        judge_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ),
        return_tensors="pt",
    ).to(judge_model.device)
    prompt_length = inputs["input_ids"].shape[1]

    def allowed_tokens(batch_id, current_ids):
        generated = current_ids[prompt_length:].tolist()
        allowed = set()
        for tokens in score_tokens.values():
            if tokens[:len(generated)] == generated:
                allowed.add(
                    tokens[len(generated)]
                    if len(generated) < len(tokens)
                    else judge_tokenizer.eos_token_id
                )
        return list(allowed)

    with torch.inference_mode():
        output = judge_model.generate(
            **inputs,
            max_new_tokens=max(map(len, score_tokens.values())) + 1,
            do_sample=False,
            prefix_allowed_tokens_fn=allowed_tokens,
            pad_token_id=judge_tokenizer.eos_token_id,
        )

    text = judge_tokenizer.decode(
        output[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return float(text)


retrieval_prompt = """다음 청크들만으로 질문의 확인 항목에 직접 답할 수 있는지 평가하세요.
질문에 나온 사업과 다른 사업의 정보는 인정하지 마세요.

1: 모든 항목의 근거가 있음
0.5: 일부 항목의 근거만 있음
0: 어떤 항목의 근거도 없음

질문: {query}
확인 항목: {checks}
검색된 청크:
{context}

점수만 출력하세요."""

answer_eval_prompt = """검색된 청크를 기준으로 이미 생성된 답변을 평가하세요.
질문의 확인 항목을 빠짐없이 답했는지, 답변 내용이 청크와 일치하는지 확인하세요.
새로운 답변을 작성하지 마세요.

1: 모든 항목에 정확히 답하고 근거도 있음
0.5: 일부 항목만 정확히 답함
0: 답하지 못했거나 핵심 내용이 틀림

질문: {query}
확인 항목: {checks}
검색된 청크:
{context}
생성된 답변:
{answer}

점수만 출력하세요."""

rows = []
for item, result in zip(evaluation_data, answers):
    if item["query"] != result["query"]:
        raise ValueError("질문과 생성 답변의 순서가 다릅니다.")

    query = result["query"]
    checks = ", ".join(item["checks"])
    context = "\n\n".join(result["contexts"])

    retrieval_score = judge_score(retrieval_prompt.format(
        query=query, checks=checks, context=context
    ))
    answer_score = judge_score(answer_eval_prompt.format(
        query=query, checks=checks, context=context, answer=result["answer"]
    ))
    rows.append({
        "query": query,
        "retrieval_score": retrieval_score,
        "answer_score": answer_score,
    })

evaluation_df = pd.DataFrame(rows)
display(evaluation_df)
print("평균")
display(evaluation_df[["retrieval_score", "answer_score"]].mean())
